# Notebook 2: ObliQA Main Results

Reproduces Table 2 of the paper.

**Systems evaluated:**
- BGE only
- BM25 only  
- BGE + CE + Window (no fusion)
- Full pipeline (BM25 + BGE + RRF + CE + Window)

**Metrics:** Recall@1, Recall@5, Recall@10, MAP@10

**Dataset:** ObliQA test split (n=2,786)

**Runtime:** ~90 min on T4 GPU (all 4 systems)

In [ ]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 nltk

import pickle, json, string, requests, time
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from nltk.tokenize import sent_tokenize
import faiss, nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
# Load pre-built index (from Notebook 1 output)
# OR rebuild from scratch
REBUILD = False  # Set True if index not available

if REBUILD:
    exec(open('01_setup_and_indexing.py').read())
else:
    print('Loading pre-built ObliQA index...')
    clean_passages = pickle.load(open('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/clean_passages_obliqua.pkl', 'rb'))
    index          = faiss.read_index('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/faiss_index_obliqua.bin')
    bm25           = pickle.load(open('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/bm25_obliqua.pkl', 'rb'))
    test_data      = json.loads(requests.get(
        'https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/ObliQA_test.json'
    ).text)
    bge_model      = SentenceTransformer('BAAI/bge-base-en-v1.5')
    cross_encoder  = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    print(f'Passages: {len(clean_passages)}, Index: {index.ntotal}, Questions: {len(test_data)}')

In [ ]:
# Core helper functions
def tokenize(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

def inner_decay(sentences, ce_scores, threshold=0.0, delta=2.0):
    if not sentences:
        return []
    scored = sorted(zip(ce_scores, sentences), reverse=True)
    selected = []
    for i, (score, sent) in enumerate(scored):
        if score < threshold:
            break
        if i > 0 and (scored[i-1][0] - score) > delta:
            break
        selected.append((score, sent))
    return selected if selected else [scored[0]]

def get_window(doc_id, passage_idx):
    doc_passages = [(i, p) for i, p in enumerate(clean_passages)
                    if p['DocumentID'] == doc_id]
    current_pos  = next(
        (pos for pos, (i, p) in enumerate(doc_passages) if i == passage_idx), None
    )
    if current_pos is None:
        return clean_passages[passage_idx]['Passage']
    window = []
    for offset in [-1, 0, 1]:
        pos = current_pos + offset
        if 0 <= pos < len(doc_passages):
            window.append(doc_passages[pos][1]['Passage'])
    return ' '.join(window)

def build_pid_to_idx():
    pid_to_idx = {}
    for i, p in enumerate(clean_passages):
        pid = (p['DocumentID'], p['PassageID'])
        if pid not in pid_to_idx:
            pid_to_idx[pid] = i
    return pid_to_idx

pid_to_idx = build_pid_to_idx()
print('Helper functions ready.')

In [ ]:
# Retrieval functions
def retrieve_bge_only(query, top_n=10):
    qe = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(qe, top_n)
    results, seen = [], set()
    for i, idx in enumerate(indices[0]):
        p   = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        if pid not in seen:
            seen.add(pid)
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID']})
    return results

def retrieve_bm25_only(query, top_n=10):
    tokens      = tokenize(query)
    bm25_scores = bm25.get_scores(tokens)
    top_indices = np.argsort(bm25_scores)[::-1][:top_n]
    results, seen = [], set()
    for idx in top_indices:
        p   = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        if pid not in seen:
            seen.add(pid)
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID']})
    return results

def retrieve_bge_ce_window(query, top_n=100):
    """BGE + CE + Window, no RRF fusion."""
    qe = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(qe, top_n)
    results, seen = [], set()
    for rank_idx in range(min(top_n, len(indices[0]))):
        idx = indices[0][rank_idx]
        p   = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        if pid in seen:
            continue
        seen.add(pid)
        curr_sents = sent_tokenize(p['Passage'])
        if not curr_sents:
            continue
        curr_ce  = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce   = float(max(curr_ce))
        window   = get_window(p['DocumentID'], idx)
        win_sents = sent_tokenize(window)
        win_ce    = cross_encoder.predict([[query, s] for s in win_sents])
        kept      = inner_decay(win_sents, win_ce)
        if kept:
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'],
                            'top_ce_score': top_ce})
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

def retrieve_full_pipeline(query, top_n=100, rrf_k=60):
    """Full pipeline: BM25 + BGE RRF + CE + Window."""
    # BGE
    qe = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    _, bge_indices = index.search(qe, top_n)
    bge_indices    = bge_indices[0]
    # BM25
    bm25_scores    = bm25.get_scores(tokenize(query))
    bm25_indices   = np.argsort(bm25_scores)[::-1][:top_n]
    # RRF
    rrf_scores = {}
    for rank, idx in enumerate(bge_indices):
        p   = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1/(rrf_k + rank + 1)
    for rank, idx in enumerate(bm25_indices):
        p   = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1/(rrf_k + rank + 1)
    sorted_pids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:50]
    # CE + Window
    results, seen = [], set()
    for pid, _ in sorted_pids:
        if pid in seen:
            continue
        seen.add(pid)
        idx = pid_to_idx.get(pid)
        if idx is None:
            continue
        p          = clean_passages[idx]
        curr_sents = sent_tokenize(p['Passage'])
        if not curr_sents:
            continue
        curr_ce  = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce   = float(max(curr_ce))
        window   = get_window(p['DocumentID'], idx)
        win_sents = sent_tokenize(window)
        win_ce    = cross_encoder.predict([[query, s] for s in win_sents])
        kept      = inner_decay(win_sents, win_ce)
        if kept:
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'],
                            'top_ce_score': top_ce})
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

print('Retrieval functions defined.')

In [ ]:
def evaluate_obliqua(test_data, retrieve_fn, top_n=10):
    recall_1, recall_5, recall_10, map_10 = [], [], [], []
    for item in tqdm(test_data):
        query   = item['Question']
        gt_pids = set((p['DocumentID'], p['PassageID']) for p in item['Passages'])
        results   = retrieve_fn(query, top_n=top_n)
        retrieved = [(r['DocumentID'], r['PassageID']) for r in results]
        recall_1.append(1.0 if any(p in gt_pids for p in retrieved[:1]) else 0.0)
        recall_5.append(1.0 if any(p in gt_pids for p in retrieved[:5]) else 0.0)
        recall_10.append(1.0 if any(p in gt_pids for p in retrieved[:10]) else 0.0)
        ap, hits = 0.0, 0
        for k, pid in enumerate(retrieved[:10], 1):
            if pid in gt_pids:
                hits += 1
                ap   += hits / k
        map_10.append(ap / max(len(gt_pids), 1))
    return {
        'Recall@1':  np.mean(recall_1),
        'Recall@5':  np.mean(recall_5),
        'Recall@10': np.mean(recall_10),
        'MAP@10':    np.mean(map_10)
    }
print('Evaluator ready.')

In [ ]:
# Run all systems
all_results = {}

print('1/4 BGE only (n=2786)...')
all_results['BGE only'] = evaluate_obliqua(test_data, retrieve_bge_only, top_n=10)
print(all_results['BGE only'])

print('\n2/4 BM25 only (n=2786)...')
all_results['BM25 only'] = evaluate_obliqua(test_data, retrieve_bm25_only, top_n=10)
print(all_results['BM25 only'])

print('\n3/4 BGE + CE + Window (n=2786)...')
all_results['BGE + CE + Window'] = evaluate_obliqua(test_data, retrieve_bge_ce_window, top_n=100)
print(all_results['BGE + CE + Window'])

print('\n4/4 Full pipeline (n=2786)...')
all_results['Full pipeline'] = evaluate_obliqua(test_data, retrieve_full_pipeline, top_n=100)
print(all_results['Full pipeline'])

In [ ]:
# Print Table 2
print('\n=== TABLE 2: ObliQA Results (n=2,786) ===')
print(f'{"System":<30} {"R@1":>7} {"R@5":>7} {"R@10":>7} {"MAP@10":>8}')
print('-' * 65)
for system, metrics in all_results.items():
    print(f'{system:<30} {metrics["Recall@1"]:>7.4f} {metrics["Recall@5"]:>7.4f} '
          f'{metrics["Recall@10"]:>7.4f} {metrics["MAP@10"]:>8.4f}')

# Save
import json
with open('/kaggle/working/obliqua_main_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('\nSaved: obliqua_main_results.json')